# Testing new artifact rejection methods

In [ ]:
import os
import re
import sys
import json
import logging
import numpy as np
import scipy.io as scio
import argparse
import shutil
from datetime import datetime
import spikeinterface.full as si
import matplotlib.pyplot as plt

In [6]:
# set up session
catgt_directory = os.path.abspath('D:\\Temp\\catgt_M671_2026_08_27_g0')
probe_idx = 0
SI_NUM_JOBS = 14
CHUNK_DURATION = '1s'

In [4]:
# load in the data
raw_rec = si.read_spikeglx(catgt_directory, stream_name=f"imec{probe_idx}.ap")
raw_rec

SpikeGLXRecordingExtractor: 1536 channels - 30.0kHz - 1 segments - 193,185,465 samples 
                            6,439.52s (1.79 hours) - int16 dtype - 552.71 GiB

In [5]:
# Get probe information as a DataFrame
raw_rec.get_probe().to_dataframe()

,x,y,contact_shapes,width,shank_ids,contact_ids
0,0.0,0.0,square,12.0,0,s0e0
1,32.0,0.0,square,12.0,0,s0e1
2,0.0,15.0,square,12.0,0,s0e2
3,32.0,15.0,square,12.0,0,s0e3
4,0.0,30.0,square,12.0,0,s0e4
...,...,...,...,...,...,...
1531,782.0,2835.0,square,12.0,3,s3e379
1532,750.0,2850.0,square,12.0,3,s3e380
1533,782.0,2850.0,square,12.0,3,s3e381
1534,750.0,2865.0,square,12.0,3,s3e382


In [10]:
def detect_artifacts(rec, probe_idx):
    tag = f"probe {probe_idx}"
    print(f"Detecting artifacts for {tag}...")

    try:
        ap_channel_ids = rec.channel_ids[:1536]   # drop the trailing 4 SY sync channels
        rec_ap = rec.select_channels(channel_ids=ap_channel_ids)
        method_kwargs = dict(proportion=0.2, saturation_threshold_uV=1200) # default proportion is 0.2. tried 0.8 and it missed a bunch of the artifacts
        job_kwargs = dict(n_jobs=SI_NUM_JOBS, chunk_duration=CHUNK_DURATION, progress_bar=True)
        artifacts = si.detect_artifact_periods(rec_ap, method='saturation', 
                                               method_kwargs=method_kwargs, job_kwargs = job_kwargs)

        print(f"Detected {len(artifacts)} artifacts for {tag}")
        return artifacts

    except Exception as e:
        print(f"Exception detecting artifacts for {tag}: {e}")
        return []

In [11]:
# detect artifacts
artifact_periods = detect_artifacts(raw_rec, probe_idx)

Detecting artifacts for probe 0...


detect saturation artifacts (workers: 14 processes spawn):   0%|          | 0/6440 [00:00<?, ?it/s]

Detected 1186 artifacts for probe 0


In [13]:
np.set_printoptions(threshold=np.inf)
print(artifact_periods)

[(0,    100028,    100106) (0,    100108,    100111)
 (0,    675710,    675714) (0,   6841813,   6841853)
 (0,   6842014,   6842066) (0,   8033916,   8033962)
 (0,   8471184,   8471186) (0,   8471188,   8471286)
 (0,   8471288,   8471292) (0,  10098999,  10099031)
 (0,  10099200,  10099252) (0,  11981050,  11981051)
 (0,  11981053,  11981159) (0,  11981392,  11981393)
 (0,  11981399,  11981402) (0,  13137773,  13137774)
 (0,  13137777,  13137875) (0,  13313476,  13314008)
 (0,  14173639,  14173671) (0,  14173840,  14173892)
 (0,  17160629,  17160720) (0,  17497227,  17497327)
 (0,  17540148,  17540150) (0,  17540156,  17540157)
 (0,  17540160,  17540170) (0,  17540537,  17540689)
 (0,  19126716,  19126739) (0,  19126916,  19126968)
 (0,  19525903,  19525943) (0,  19525947,  19525950)
 (0,  19526102,  19526155) (0,  20343739,  20343807)
 (0,  20849944,  20850080) (0,  20931007,  20931037)
 (0,  20931207,  20931260) (0,  22813631,  22813697)
 (0,  22813701,  22813702) (0,  22838747,  228

In [15]:
print(type(artifact_periods), len(artifact_periods))
if len(artifact_periods):
    print(artifact_periods[0], type(artifact_periods[0]))

<class 'numpy.ndarray'> 1186
(0, 100028, 100106) <class 'numpy.void'>


In [16]:
artifact_periods.dtype.names

('segment_index', 'start_sample_index', 'end_sample_index')

In [17]:
starts = artifact_periods['start_sample_index']   # use whatever dtype.names actually shows
ends = artifact_periods['end_sample_index']
durations = ends - starts
av = np.mean(durations)
print(f"Average artifact duration: {av} samples ({av/raw_rec.sampling_frequency:.3f} seconds)")

Average artifact duration: 54.381112984822934 samples (0.002 seconds)


In [19]:
print(starts/raw_rec.sampling_frequency)

[3.33426667e+00 3.33693333e+00 2.25236667e+01 2.28060433e+02
 2.28067133e+02 2.67797200e+02 2.82372800e+02 2.82372933e+02
 2.82376267e+02 3.36633300e+02 3.36640000e+02 3.99368333e+02
 3.99368433e+02 3.99379733e+02 3.99379967e+02 4.37925767e+02
 4.37925900e+02 4.43782533e+02 4.72454633e+02 4.72461333e+02
 5.72020967e+02 5.83240900e+02 5.84671600e+02 5.84671867e+02
 5.84672000e+02 5.84684567e+02 6.37557200e+02 6.37563867e+02
 6.50863433e+02 6.50864900e+02 6.50870067e+02 6.78124633e+02
 6.94998133e+02 6.97700233e+02 6.97706900e+02 7.60454367e+02
 7.60456700e+02 7.61291567e+02 7.64166867e+02 7.64168533e+02
 7.64168700e+02 7.64171433e+02 7.96979833e+02 7.96979967e+02
 8.32835700e+02 8.33219267e+02 8.64234600e+02 8.64245167e+02
 8.64263267e+02 8.64269367e+02 8.82976533e+02 8.82976700e+02
 8.82980967e+02 8.82984667e+02 9.43173500e+02 9.44206167e+02
 9.46881633e+02 9.46987933e+02 9.56528367e+02 9.78657400e+02
 9.78661833e+02 1.00649583e+03 1.00649603e+03 1.00649623e+03
 1.01783247e+03 1.017841

In [20]:
def detect_artifacts(rec, probe_idx):
    tag = f"probe {probe_idx}"
    print(f"Detecting artifacts for {tag}...")

    try:
        ap_channel_ids = rec.channel_ids[:1536]   # drop the trailing 4 SY sync channels
        rec_ap = rec.select_channels(channel_ids=ap_channel_ids)
        method_kwargs = dict(proportion=0.25, saturation_threshold_uV=1200) # default proportion is 0.2. tried 0.8 and it missed a bunch of the artifacts
        job_kwargs = dict(n_jobs=SI_NUM_JOBS, chunk_duration=CHUNK_DURATION, progress_bar=True)
        artifacts = si.detect_artifact_periods(rec_ap, method='saturation', 
                                               method_kwargs=method_kwargs, job_kwargs = job_kwargs)

        print(f"Detected {len(artifacts)} artifacts for {tag}")
        return artifacts

    except Exception as e:
        print(f"Exception detecting artifacts for {tag}: {e}")
        return []

In [21]:
# detect artifacts
artifact_periods = detect_artifacts(raw_rec, probe_idx)

Detecting artifacts for probe 0...


detect saturation artifacts (workers: 14 processes spawn):   0%|          | 0/6440 [00:00<?, ?it/s]

Detected 126 artifacts for probe 0


In [22]:
np.set_printoptions(threshold=np.inf)
print(artifact_periods)

[(0,    675710,    675713) (0,   6841813,   6841852)
 (0,   6842014,   6842066) (0,  10098999,  10099029)
 (0,  10099200,  10099252) (0,  14173639,  14173670)
 (0,  14173840,  14173892) (0,  19126716,  19126737)
 (0,  19126916,  19126968) (0,  19525903,  19525942)
 (0,  19526103,  19526155) (0,  20931007,  20931035)
 (0,  20931207,  20931259) (0,  26489603,  26489608)
 (0,  28406449,  28406495) (0,  28409638,  28409649)
 (0,  29359722,  29359748) (0,  29359856,  29359907)
 (0,  39105436,  39105438) (0,  39105440,  39105441)
 (0,  39105443,  39105448) (0,  44521353,  44521452)
 (0,  44521455,  44521456) (0,  44521459,  44521460)
 (0,  44521462,  44521672) (0,  44521674,  44521675)
 (0,  44521818,  44521932) (0,  56214634,  56214667)
 (0,  56217630,  56217641) (0,  58402617,  58402673)
 (0,  58402774,  58402811) (0,  58402813,  58402830)
 (0,  62331248,  62331252) (0,  62331254,  62331291)
 (0,  62331294,  62331295) (0,  64667888,  64667921)
 (0,  64670996,  64671007) (0,  65147627,  651

In [23]:
starts = artifact_periods['start_sample_index']   # use whatever dtype.names actually shows
ends = artifact_periods['end_sample_index']
durations = ends - starts
av = np.mean(durations)
print(f"Average artifact duration: {av} samples ({av/raw_rec.sampling_frequency:.3f} seconds)")

Average artifact duration: 33.54761904761905 samples (0.001 seconds)


In [25]:
total = np.sum(durations)
print(f"Total artifact duration: {total} samples ({total/raw_rec.sampling_frequency:.3f} seconds)")

Total artifact duration: 4227 samples (0.141 seconds)


In [26]:
ar_rec = si.silence_periods(raw_rec, artifact_periods, mode='zeros')

In [28]:
ar_rec

SilencedPeriodsRecording: 1536 channels - 30.0kHz - 1 segments - 193,185,465 samples 
                          6,439.52s (1.79 hours) - int16 dtype - 552.71 GiB

# big problem
Cannot reject artifact periods from LFP data after downsampling